In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/content/amazon_grocery.csv", low_memory=False)

In [ ]:
df.to_csv('final_data.csv', index=False)

In [ ]:
df.columns

Index(['name', 'price', 'url', 'keyword', 'Brand Name', 'Flavor',
       'Allergen Information', 'Item Weight', 'Item Form', 'Unit Count',
       ...
       'Sheet Count', 'Package Type', 'Manufacturer Food Processing Method',
       'Fastener Type', 'Metal Type', 'Thread Type', 'Inside Thread Size',
       'Genre', 'Edition', 'Are batteries required?'],
      dtype='object', length=101)

In [ ]:
df.head()

,name,price,url,keyword,Brand Name,Flavor,Allergen Information,Item Weight,Item Form,Unit Count,...,Sheet Count,Package Type,Manufacturer Food Processing Method,Fastener Type,Metal Type,Thread Type,Inside Thread Size,Genre,Edition,Are batteries required?
0,Juhayna,257,https://www.amazon.eg/-/en/JUHAYNA-Carton-Crea...,milk,Juhayna,Full Cream,Dairy,1 Kilograms,Liquid,1 Liters,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Almarai,49,https://www.amazon.eg/-/en/Almarai-plain-milk-...,milk,Almarai,Without Flavor,Milk,1 Kilograms,Liquid,1000 Milliliters,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Juhayna,515,https://www.amazon.eg/-/en/JUHAYNA-Carton-Crea...,milk,Juhayna,Full Cream,Dairy,2 Kilograms,Liquid,2 Liters,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Almarai,302,https://www.amazon.eg/-/en/Almarai-Milk-Full-F...,milk,Almarai,coffee,Milk,1 Kilograms,Liquid,1 Liters,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Almarai,49,https://www.amazon.eg/-/en/Almarai-Plain-Full-...,milk,Almarai,unflavored,Milk,1 Kilograms,Liquid,1000 Count,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
cols = ['name', 'price', 'keyword', 'Brand Name', 'Flavor',
        'Allergen Information', 'Item Weight', 'Item Form', 'Unit Count']
df = df[cols]

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9405 entries, 0 to 9404
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   name                  9405 non-null   object
 1   price                 9026 non-null   object
 2   keyword               9405 non-null   object
 3   Brand Name            9096 non-null   object
 4   Flavor                5750 non-null   object
 5   Allergen Information  1016 non-null   object
 6   Item Weight           5498 non-null   object
 7   Item Form             5174 non-null   object
 8   Unit Count            4145 non-null   object
dtypes: object(9)
memory usage: 661.4+ KB


In [ ]:
print(df.head())

      name price keyword Brand Name          Flavor Allergen Information  \
0  Juhayna   257    milk    Juhayna      Full Cream                Dairy   
1  Almarai    49    milk    Almarai  Without Flavor                 Milk   
2  Juhayna   515    milk    Juhayna      Full Cream                Dairy   
3  Almarai   302    milk    Almarai          coffee                 Milk   
4  Almarai    49    milk    Almarai      unflavored                 Milk   

   Item Weight Item Form        Unit Count  
0  1 Kilograms    Liquid          1 Liters  
1  1 Kilograms    Liquid  1000 Milliliters  
2  2 Kilograms    Liquid          2 Liters  
3  1 Kilograms    Liquid          1 Liters  
4  1 Kilograms    Liquid        1000 Count  


**my target is to determine the price so Allergen feature is not important**

In [ ]:
df = df.drop(columns=['Allergen Information'])

In [ ]:
df.columns

Index(['name', 'price', 'keyword', 'Brand Name', 'Flavor', 'Item Weight',
       'Item Form', 'Unit Count'],
      dtype='object')

**preprocess data..It includes three phase:
1.firstly:Clean text data**

In [ ]:
text_cols = ['name','keyword','Brand Name','Flavor','Item Form']

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

**Clean numerical features**

In [ ]:
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['price'])

In [ ]:
df['price']

,price
0,257.0
1,49.0
2,515.0
3,302.0
4,49.0
...,...
9399,299.0
9400,244.0
9401,404.0
9402,250.0


**ItemWeight cleaning**

In [ ]:
def convert_weight(x):
    if pd.isna(x):
        return np.nan

    x = str(x).lower().strip()

    try:
        import re
        numbers = re.findall(r'(\d+\.?\d*)', x)
        if not numbers:
            return np.nan

        value = float(numbers[0])

        if 'kilogram' in x or 'kg' in x or 'كيلو' in x:
            return value
        elif 'gram' in x or 'g ' in x or 'جرام' in x:
            return value / 1000
        elif 'liter' in x or 'l ' in x or 'لتر' in x:
            return value
        elif 'milliliter' in x or 'ml' in x:
            return value / 1000
        elif 'count' in x or 'عدد' in x:

            return np.nan
        else:

            return value if value < 20 else np.nan
    except:
        return np.nan

**Handle outliers in target**

In [ ]:

def handle_price_outliers(df, column='price'):

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    print(f"number of ouliers {len(outliers)}")
    print(f"acceptable range {lower_bound:.2f} - {upper_bound:.2f}")

    df_clean = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]



    return df_clean

df = handle_price_outliers(df)

number of ouliers 556
acceptable range -215.00 - 529.00


In [ ]:
# أضف هذا قبل OneHotEncoder
def standardize_categories(df):



    brand_mapping = {
        'juhayna': ['juhayna', 'جهينا'],
        'almarai': ['almarai', 'المراعي'],
        'lamar': ['lamar', 'لامار'],
        'nido': ['nido', 'نيدو'],
        'nestle': ['nestle', 'nestlé', 'نستله'],
        'lipton': ['lipton', 'ليبتون'],
        'ahmad tea': ['ahmad', 'أحمد', 'ahmad tea'],
        'crystal': ['crystal', 'كريستال'],
        'afia': ['afia', 'عفيا'],
        'abu auf': ['abu auf', 'ابو عوف'],
        'purity': ['purity', 'بيورتي'],
        'spicekick': ['spicekick', 'spice kick'],
    }

    def map_brand(brand):
        if pd.isna(brand):
            return 'unknown'
        brand_lower = str(brand).lower()
        for std_brand, variants in brand_mapping.items():
            if any(variant in brand_lower for variant in variants):
                return std_brand
        return brand_lower if len(brand_lower) > 2 else 'unknown'

    df['Brand Name'] = df['Brand Name'].apply(map_brand)

    flavor_mapping = {
        'plain': ['plain', 'original', 'natural', 'unflavored', 'without flavor', 'بدون نكهة'],
        'chocolate': ['chocolate', 'شوكولاتة', 'cocoa', 'كاكاو'],
        'strawberry': ['strawberry', 'فراولة'],
        'vanilla': ['vanilla', 'فانيليا'],
        'mango': ['mango', 'مانجو'],
        'full cream': ['full cream', 'full fat', 'كامل الدسم'],
        'skimmed': ['skimmed', 'skim', 'منزوع الدسم'],
        'coffee': ['coffee', 'قهوة'],
        'coconut': ['coconut', 'جوز الهند'],
        'almond': ['almond', 'لوز'],
    }

    def map_flavor(flavor):
        if pd.isna(flavor):
            return 'unknown'
        flavor_lower = str(flavor).lower()
        for std_flavor, variants in flavor_mapping.items():
            if any(variant in flavor_lower for variant in variants):
                return std_flavor
        return flavor_lower

    df['Flavor'] = df['Flavor'].apply(map_flavor)


    form_mapping = {
        'liquid': ['liquid', 'سائل', 'drink', 'beverage'],
        'powder': ['powder', 'مسحوق', 'بودرة', 'ground'],
        'solid': ['solid', 'صلب', 'block', 'bar'],
        'cream': ['cream', 'كريم', 'spread'],
        'granule': ['granule', 'gravel', 'حبيبات'],
    }

    def map_form(form):
        if pd.isna(form):
            return 'unknown'
        form_lower = str(form).lower()
        for std_form, variants in form_mapping.items():
            if any(variant in form_lower for variant in variants):
                return std_form
        return form_lower

    df['Item Form'] = df['Item Form'].apply(map_form)

    return df

df = standardize_categories(df)

In [ ]:

def extract_features_from_name(df):



    df['is_organic'] = df['name'].str.contains(r'\borganic\b|\bعضوي\b', case=False, na=False).astype(int)
    df['is_sugar_free'] = df['name'].str.contains(r'\bsugar[-\s]?free\b|\bno sugar\b|\bبدون سكر\b', case=False, na=False).astype(int)
    df['is_gluten_free'] = df['name'].str.contains(r'\bgluten[-\s]?free\b|\bخالي من الجلوتين\b', case=False, na=False).astype(int)
    df['is_vegan'] = df['name'].str.contains(r'\bvegan\b|\bنباتي\b', case=False, na=False).astype(int)
    df['is_keto'] = df['name'].str.contains(r'\bketo\b|\bكيتو\b', case=False, na=False).astype(int)
    df['is_diet'] = df['name'].str.contains(r'\bdiet\b|\bدايت\b|\blight\b', case=False, na=False).astype(int)
    df['is_zero'] = df['name'].str.contains(r'\bzero\b|\bصفر\b', case=False, na=False).astype(int)
    df['is_stevia'] = df['name'].str.contains(r'\bstevia\b|\bستيفيا\b', case=False, na=False).astype(int)

    # استخراج حجم العبوة إذا كان موجوداً
    df['pack_size'] = df['name'].str.extract(r'(\d+)\s*(?:piece|pcs|قطعة|pack)').astype(float)


    for col in ['is_organic', 'is_sugar_free', 'is_gluten_free', 'is_vegan', 'is_keto', 'is_diet', 'is_zero', 'is_stevia']:
        print(f"  {col}: {df[col].sum()} منتج")

    return df

df = extract_features_from_name(df)

  is_organic: 80 منتج
  is_sugar_free: 104 منتج
  is_gluten_free: 91 منتج
  is_vegan: 46 منتج
  is_keto: 98 منتج
  is_diet: 240 منتج
  is_zero: 61 منتج
  is_stevia: 152 منتج


In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack, csr_matrix
def create_tfidf_features(df, text_column='name', max_features=500):



    def clean_text(text):
        if pd.isna(text):
            return ""
        text = str(text).lower()

        text = re.sub(r'http\S+', '', text)

        text = re.sub(r'[^\w\s\u0600-\u06ff]', ' ', text)

        text = re.sub(r'\d+', '', text)

        words = text.split()
        words = [w for w in words if len(w) > 2]
        return ' '.join(words)

    df['clean_name'] = df[text_column].apply(clean_text)


    tfidf = TfidfVectorizer(
        max_features=max_features,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95
    )

    name_features = tfidf.fit_transform(df['clean_name'])
    print(f"num of {name_features.shape[1]} features ")

    return name_features, tfidf, df

name_features, tfidf, df = create_tfidf_features(df, max_features=500)

num of 500 features 


**Clean the UnitCount Feature**

In [ ]:
df['Unit Count'] = pd.to_numeric(df['Unit Count'], errors='coerce')


In [ ]:
X = df.drop('price', axis=1)   # features
y = df['price']                # target

In [ ]:
print(X.head())


      name keyword Brand Name      Flavor  Item Weight Item Form  Unit Count  \
0  juhayna    milk    juhayna  full cream  1 Kilograms    liquid         NaN   
1  almarai    milk    almarai       plain  1 Kilograms    liquid         NaN   
2  juhayna    milk    juhayna  full cream  2 Kilograms    liquid         NaN   
3  almarai    milk    almarai      coffee  1 Kilograms    liquid         NaN   
4  almarai    milk    almarai       plain  1 Kilograms    liquid         NaN   

   is_organic  is_sugar_free  is_gluten_free  is_vegan  is_keto  is_diet  \
0           0              0               0         0        0        0   
1           0              0               0         0        0        0   
2           0              0               0         0        0        0   
3           0              0               0         0        0        0   
4           0              0               0         0        0        0   

   is_zero  is_stevia  pack_size clean_name  
0        0      

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer

In [ ]:
print(y.head())

0    257.0
1     49.0
2    515.0
3    302.0
4     49.0
Name: price, dtype: float64


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8071 entries, 0 to 9404
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            8071 non-null   object 
 1   price           8071 non-null   float64
 2   keyword         8071 non-null   object 
 3   Brand Name      8071 non-null   object 
 4   Flavor          8071 non-null   object 
 5   Item Weight     4854 non-null   object 
 6   Item Form       8071 non-null   object 
 7   Unit Count      0 non-null      float64
 8   is_organic      8071 non-null   int64  
 9   is_sugar_free   8071 non-null   int64  
 10  is_gluten_free  8071 non-null   int64  
 11  is_vegan        8071 non-null   int64  
 12  is_keto         8071 non-null   int64  
 13  is_diet         8071 non-null   int64  
 14  is_zero         8071 non-null   int64  
 15  is_stevia       8071 non-null   int64  
 16  pack_size       485 non-null    float64
 17  clean_name      8071 non-null   object

In [ ]:
df = df.drop(columns=['Unit Count'])

In [ ]:
df.shape

(8071, 17)

In [ ]:
df.columns

Index(['name', 'price', 'keyword', 'Brand Name', 'Flavor', 'Item Weight',
       'Item Form', 'is_organic', 'is_sugar_free', 'is_gluten_free',
       'is_vegan', 'is_keto', 'is_diet', 'is_zero', 'is_stevia', 'pack_size',
       'clean_name'],
      dtype='object')

**To Imput Itemweight values**

In [ ]:
df['Item Weight'] = df['Item Weight'].fillna(df['Item Weight'].median())

TypeError: Cannot convert ['1 Kilograms' '1 Kilograms' '2 Kilograms' ... '400 Grams' '200 Grams'
 '340 Grams'] to numeric

In [ ]:
df['Item Weight']

,Item Weight
0,1 Kilograms
1,1 Kilograms
2,2 Kilograms
3,1 Kilograms
4,1 Kilograms
...,...
9399,454 Grams
9400,125 Grams
9401,400 Grams
9402,200 Grams


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8071 entries, 0 to 9404
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            8071 non-null   object 
 1   price           8071 non-null   float64
 2   keyword         8071 non-null   object 
 3   Brand Name      8071 non-null   object 
 4   Flavor          8071 non-null   object 
 5   Item Weight     4854 non-null   object 
 6   Item Form       8071 non-null   object 
 7   is_organic      8071 non-null   int64  
 8   is_sugar_free   8071 non-null   int64  
 9   is_gluten_free  8071 non-null   int64  
 10  is_vegan        8071 non-null   int64  
 11  is_keto         8071 non-null   int64  
 12  is_diet         8071 non-null   int64  
 13  is_zero         8071 non-null   int64  
 14  is_stevia       8071 non-null   int64  
 15  pack_size       485 non-null    float64
 16  clean_name      8071 non-null   object 
dtypes: float64(2), int64(8), object(7)
mem

In [ ]:
df.shape

(8071, 17)

**Done....Step2 Our model do not recognize text using TF_IDF for nameofproduct**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=500)

name_features = tfidf.fit_transform(df['name'])

**OneHot encoding for other textual features**

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_cols = ['keyword','Brand Name','Flavor','Item Form']

encoder = OneHotEncoder()

cat_features = encoder.fit_transform(df[cat_cols])

**numerical features....So,we distribute it into text features>>Tf_IDF...cat features>>one hot...numerical features**

In [ ]:
num_features = df[['Item Weight']]

**Combine all features**

In [ ]:
df.to_csv("cleaning_amazon_grocery.csv", index=False)

In [ ]:

df['Item Weight'] = df['Item Weight'].apply(lambda x: x if x < 20 else df['Item Weight'].median())

TypeError: '<' not supported between instances of 'str' and 'int'

In [ ]:


import pandas as pd
import numpy as np
import re

df = pd.read_csv('cleaning_amazon_grocery.csv')
print(f" num of {len(df)} rows")
print(f" unique val: {df['Brand Name'].nunique()}")

df['Brand Name'] = df['Brand Name'].replace(['nan', 'none', 'null', 'unknown', ''], np.nan)

def extract_brand_from_name(row):
    if pd.notna(row['Brand Name']):
        return row['Brand Name']

    name = str(row['name']).lower()

    brand_keywords = {
        'juhayna': ['juhayna', 'جهينا'],
        'almarai': ['almarai', 'المراعي'],
        'lamar': ['lamar', 'لامار'],
        'nido': ['nido', 'نيدو'],
        'nestle': ['nestle', 'nestlé', 'نستله'],
        'lipton': ['lipton', 'ليبتون'],
        'ahmad tea': ['ahmad', 'أحمد', 'ahmad tea'],
        'crystal': ['crystal', 'كريستال'],
        'afia': ['afia', 'عفيا'],
        'abu auf': ['abu auf', 'ابو عوف', 'abu auf'],
        'purity': ['purity', 'بيورتي'],
        'spicekick': ['spicekick', 'spice kick'],
        'hero': ['hero', 'هيرو'],
        'vitrac': ['vitrac', 'فيتراك'],
        'mero': ['mero', 'ميرو'],
        'el bawadi': ['el bawadi', 'البوادي'],
        'isis': ['isis', 'ايزيس'],
        'imtenan': ['imtenan', 'امتنان'],
        'zary': ['zary', 'زري'],
        'rehana': ['rehana', 'ريحانة'],
        'dobella': ['dobella', 'دوبيلا'],
        'daria': ['daria', 'داريا'],
        'shana': ['shana', 'شنا', 'shanah'],
        'teyab': ['teyab', 'طياب'],
        'haj arafa': ['haj arafa', 'حاج عرفة'],
        'al doha': ['al doha', 'الدوحة'],
        'sedra': ['sedra', 'سدرة'],
        'zamzam': ['zamzam', 'زمزم'],
        'marvel': ['marvel', 'مارفل'],
        'tiger': ['tiger', 'تايجر'],
        'pringles': ['pringles', 'برينجلز'],
        'chipsy': ['chipsy', 'شيبسي'],
        'fox': ['fox', 'فوكس'],
        'ponky': ['ponky', 'بونكي'],
        'spuds': ['spuds', 'سبادس'],
        'raw': ['raw', 'رو'],
        'jaguar': ['jaguar', 'جاجوار'],
    }

    for brand, keywords in brand_keywords.items():
        if any(keyword in name for keyword in keywords):
            return brand


    return 'unknown'

df['Brand Name'] = df.apply(extract_brand_from_name, axis=1)


# قاموس شامل لدمج الأسماء المتشابهة
brand_mapping = {
    # العلامات التجارية الكبرى
    'juhayna': ['juhayna', 'juahyna', 'juhaina', 'جهينا', 'juhayna'],
    'almarai': ['almarai', 'al marai', 'المراعي', 'almarai'],
    'lamar': ['lamar', 'لامار', 'lamarr', 'lamar'],

    # منتجات الألبان
    'nido': ['nido', 'نيدو', 'nido'],
    'nestle': ['nestle', 'nestlé', 'نستله', 'nestle'],
    'latteria': ['latteria', 'لاتيريا', 'latteria'],
    'miro': ['miro', 'ميرو', 'miro'],
    'the milkman': ['the milkman', 'milkman', 'ذا ميلكمان'],

    # المشروبات
    'lipton': ['lipton', 'ليبتون', 'lipton'],
    'ahmad tea': ['ahmad', 'أحمد', 'ahmad tea', 'ahmed tea', 'ahmad'],
    'royal herbs': ['royal', 'royal herbs', 'royal herbs'],

    # الزيوت
    'crystal': ['crystal', 'كريستال', 'crystal'],
    'afia': ['afia', 'عفيا', 'afia'],
    'slite': ['slite', 'سلايت', 'slite'],
    'hanady': ['hanady', 'هنادي', 'hanady'],
    'helwa': ['helwa', 'حلوة', 'helwa'],

    # العلامات التجارية المصرية
    'abu auf': ['abu auf', 'ابو عوف', 'abu auf', 'abuof'],
    'purity': ['purity', 'بيورتي', 'purity'],
    'spicekick': ['spicekick', 'spice kick', 'spicekick'],
    'hero': ['hero', 'هيرو', 'hero'],
    'vitrac': ['vitrac', 'فيتراك', 'vitrac'],
    'mero': ['mero', 'ميرو', 'mero'],
    'el bawadi': ['el bawadi', 'البوادي', 'el bawadi', 'bawadi'],
    'isis': ['isis', 'ايزيس', 'isis'],
    'imtenan': ['imtenan', 'امتنان', 'imtenan'],
    'zary': ['zary', 'زري', 'zary'],
    'rehana': ['rehana', 'ريحانة', 'rehana'],
    'dobella': ['dobella', 'دوبيلا', 'dobella', 'dobella.'],
    'daria': ['daria', 'داريا', 'daria'],
    'shana': ['shana', 'شنا', 'shanah', 'shana'],
    'teyab': ['teyab', 'طياب', 'teyab'],
    'haj arafa': ['haj arafa', 'حاج عرفة', 'haj arafa'],
    'al doha': ['al doha', 'الدوحة', 'al doha', 'doha'],
    'sedra': ['sedra', 'سدرة', 'sedra'],
    'zamzam': ['zamzam', 'زمزم', 'zamzam'],
    'marvel': ['marvel', 'مارفل', 'marvel'],

    # الشيبسي والوجبات الخفيفة
    'tiger': ['tiger', 'تايجر', 'tiger'],
    'pringles': ['pringles', 'برينجلز', 'pringles'],
    'chipsy': ['chipsy', 'شيبسي', 'chipsy'],
    'fox': ['fox', 'فوكس', 'fox'],
    'ponky': ['ponky', 'بونكي', 'ponky'],
    'spuds': ['spuds', 'سبادس', 'spuds'],
    'raw': ['raw', 'رو', 'raw'],
    'jaguar': ['jaguar', 'جاجوار', 'jaguar'],
    'bonz': ['bonz', 'بونز', 'bonz'],
    'halo': ['halo', 'هالو', 'halo'],
    'freyma': ['freyma', 'فريما', 'freyma\'s', 'freymas'],

    # العلامات التجارية العالمية
    'heinz': ['heinz', 'هاينز', 'heinz'],
    'kinder': ['kinder', 'كيندر', 'kinder'],
    'ferrero': ['ferrero', 'فيريرو', 'ferrero'],
    'nutella': ['nutella', 'نوتيلا', 'nutella'],
    'cadbury': ['cadbury', 'كادبوري', 'cadbury'],
    'milka': ['milka', 'ميلكا', 'milka'],
    'lindt': ['lindt', 'ليندت', 'lindt'],
    'ritter sport': ['ritter', 'ريتر', 'ritter sport'],
    'toblerone': ['toblerone', 'توبليرون', 'toblerone'],
    'snickers': ['snickers', 'سنيكرز', 'snickers'],
    'mars': ['mars', 'مارس', 'mars'],
    'm&m': ['m&m', 'm and m', 'm&m\'s', 'm&ms'],

    # الصابون ومنتجات العناية
    'dove': ['dove', 'دوف', 'dove'],
    'lux': ['lux', 'لكس', 'lux'],
    'dettol': ['dettol', 'ديتول', 'dettol'],
    'lifebuoy': ['lifebuoy', 'لايف بوي', 'lifebuoy'],
    'pears': ['pears', 'بيرز', 'pears'],
    'camay': ['camay', 'كاماي', 'camay'],
    'palmolive': ['palmolive', 'بالموليف', 'palmolive'],
    'black lotus': ['black lotus', 'بلاك لوتس'],

    # الشامبو
    'sunsilk': ['sunsilk', 'صن سيلك', 'sunsilk'],
    'l\'oreal': ['l\'oreal', 'لوريال', 'loreal', 'l\'oreal paris'],
    'dove shampoo': ['dove', 'دوف'],
    'head & shoulders': ['head & shoulders', 'هيد اند شولدرز', 'head and shoulders'],
    'pantene': ['pantene', 'بانتين', 'pantene'],
    'tresemmé': ['tresemmé', 'تريزمي', 'tresemme'],

    # مساحيق الغسيل
    'ariel': ['ariel', 'اريال', 'ariel'],
    'tide': ['tide', 'تايد', 'tide'],
    'persil': ['persil', 'برسيل', 'persil'],
    'oxi': ['oxi', 'اوكسي', 'oxi'],

    # العصائر
    'beyti': ['beyti', 'بيتي', 'beyti'],
    'suntop': ['suntop', 'صن توب', 'suntop'],
    'stevia juice': ['stevia juice', 'ستيفيا'],
    'mogu mogu': ['mogu mogu', 'موجو موجو'],

    # الشاي
    'el arosa': ['el arosa', 'العاروضة', 'arosa'],
    'rabea': ['rabea', 'ربيعة', 'rabea'],
    'al kbous': ['al kbous', 'الكبوس', 'kbous'],

    # القهوة
    'lavazza': ['lavazza', 'لافازا', 'lavazza'],
    'illy': ['illy', 'إيلي', 'illy'],
    'nescafe': ['nescafe', 'نسكافيه', 'nescafe'],

    # الأرز والمكرونة
    'regina': ['regina', 'ريجينا', 'regina'],
    'italiano': ['italiano', 'ايطاليانو', 'italiano'],
    'granoro': ['granoro', 'جرانورو', 'granoro'],
    'barilla': ['barilla', 'باريلا', 'barilla'],

    # الدقيق
    'five stars': ['five stars', 'فايف ستارز'],
    'sonbolat el forat': ['sonbolat', 'سنبلة الفرات'],
    'biohayah': ['biohayah', 'بيو حياة'],
    'eco healthy': ['eco healthy', 'ايكو هيلثي'],

    # البسكويت
    'oreo': ['oreo', 'أوريو', 'oreo'],
    'mcvitie': ['mcvitie', 'ماكفيتيز', 'mcvities', 'mcvitie\'s'],
    'loacker': ['loacker', 'لوآكر', 'loacker'],
    'lambada': ['lambada', 'لامبادا', 'lambada'],
    'deemah': ['deemah', 'ديمة', 'deemah'],

    # العسل
    'organic nation': ['organic nation', 'أورجانيك نيشن'],
    'boosh': ['boosh', 'بوش'],
    'sinara honey': ['sinara', 'سنارة'],

    # الزبدة والفول السوداني
    'lurpak': ['lurpak', 'لورباك', 'lurpak'],
    'jif': ['jif', 'جيف', 'jif'],

    # الصلصات
    'ilou': ['ilou', 'ايلو', 'ilou'],
    'thio': ['thio', 'ثيو', 'thio'],
    'naturesta': ['naturesta', 'ناتشورستا'],
    'lavida': ['lavida', 'لافيدا'],
    'asmacup': ['asmacup', 'أسماكاب'],

    # التوابل
    'bfb': ['bfb', 'بي اف بي'],
    'bab elsham': ['bab elsham', 'باب الشام'],
    'nusayla': ['nusayla', 'نصيلة'],
    'al-qassim': ['al-qassim', 'القصيم'],

    # منتجات عامة
    'generic': ['generic', 'عام', 'nan'],
}

# دالة لتعيين العلامة التجارية الصحيحة
def map_brand(brand):
    if pd.isna(brand) or brand == 'unknown':
        return 'unknown'

    brand_lower = str(brand).lower().strip()

    # إذا كان النص قصير جداً
    if len(brand_lower) < 2:
        return 'unknown'

    # البحث في القاموس
    for standard_name, variants in brand_mapping.items():
        if any(variant in brand_lower or brand_lower in variant for variant in variants):
            return standard_name

    return brand_lower

# تطبيق التوحيد
df['Brand Name'] = df['Brand Name'].apply(map_brand)



def handle_unknown_brand(row):
    if row['Brand Name'] != 'unknown':
        return row['Brand Name']

    name = str(row['name']).lower()


    words = name.split()
    for word in words:
        if len(word) > 3 and word not in ['with', 'without', 'flavor', 'flavour', 'milk', 'juice', 'water', 'sugar']:

            if word.isalpha():
                return word

    return 'unknown'

df['Brand Name'] = df.apply(handle_unknown_brand, axis=1)






 num of 8071 rows
 unique val: 901


In [ ]:
def parse_weight(weight_value, product_name):
    """
    Attempt to extract weight in grams from the input value and product name.
    """
    if pd.isna(weight_value):
        return None

    # Convert value to string for easier handling
    weight_str = str(weight_value).strip()

    # Try to extract the number
    try:
        # Look for decimal or integer numbers in the text
        number_match = re.search(r'(\d+\.?\d*)', weight_str)
        if number_match:
            number = float(number_match.group(1))
        else:
            # If weight is just a number (from dataframe), use it directly
            number = float(weight_str)
    except ValueError:
        # If conversion to number fails (unexpected text)
        print(f"  Warning: Could not convert value '{weight_str}' to number. Will use product name.")
        number = None

    if number is None:
        return None

    # --- Guess the unit of measurement ---
    weight_in_grams = None
    product_name_lower = str(product_name).lower()

    # 1. Look for clear unit indicators in the weight value itself (weight_str)
    if 'kg' in weight_str.lower() or 'كجم' in weight_str or 'كيلو' in weight_str:
        weight_in_grams = number * 1000
    elif 'g' in weight_str.lower() and 'kg' not in weight_str.lower():  # g but not kg
        weight_in_grams = number
    elif 'ml' in weight_str.lower() or 'مل' in weight_str or 'ملى' in weight_str:
        # For liquids, 1ml ≈ 1g for unit standardization. This can be adjusted later.
        weight_in_grams = number
    elif 'l' in weight_str.lower() and 'ml' not in weight_str.lower():
        weight_in_grams = number * 1000

    # 2. If we didn't find a unit in the weight, look in the product name
    if weight_in_grams is None:
        if 'kg' in product_name_lower or 'كجم' in product_name_lower or ' كيلو ' in product_name_lower or 'كيلو' in product_name_lower:
            weight_in_grams = number * 1000
        elif 'g' in product_name_lower and 'kg' not in product_name_lower:
            weight_in_grams = number
        elif 'ml' in product_name_lower or 'مل' in product_name_lower:
            weight_in_grams = number
        elif 'l' in product_name_lower and 'ml' not in product_name_lower:
            weight_in_grams = number * 1000
        else:
            # 3. If we find no evidence, we leave it as None to handle later
            # weight_in_grams = number # This would be a bold assumption
            pass

    # 4. Handle illogical values (like weight less than 1 gram for non-spice products)
    if weight_in_grams is not None and weight_in_grams < 1 and 'spices' not in product_name_lower and 'بهارات' not in product_name_lower:
        print(f"  Illogical value: {weight_value} -> {weight_in_grams} grams. Will ignore.")
        return None

    return weight_in_grams

# Apply the function to each row
print("Analyzing weights...")
df['weight_in_grams'] = df.apply(lambda row: parse_weight(row['Item Weight'], row['name']), axis=1)

print("Weight processing complete. Invalid values are now None.")
print(f"Number of valid values in weight_in_grams: {df['weight_in_grams'].notna().sum()}")
print("Sample results:")
print(df[['name', 'Item Weight', 'weight_in_grams']].head(10))

# --- 2. Fixing the pack_size column ---
print("\n--- Step 2: Processing pack_size Column ---")

def extract_pack_size(product_name):
    """
    Attempt to extract the number of pieces in a package from the product name.
    """
    product_name_lower = str(product_name).lower()
    pack_size = None

    # List of keywords that might indicate multiple pieces
    keywords = [
        r'(\d+)\s*(pack|pieces|pcs|قطعة|عدد|pakt|عبوة|set)',
        r'(pack|pieces|pcs|قطعة|عدد|pakt|عبوة|set)\s*[:\-]?\s*(\d+)',
        r'(\d+)\s*x\s*\d+\s*(pack|pcs)'  # e.g., 12 x 25g pack
    ]

    for pattern in keywords:
        match = re.search(pattern, product_name_lower)
        if match:
            # The number might be in the first or second group depending on the pattern
            if match.group(1).isdigit():
                pack_size = int(match.group(1))
                break
            elif len(match.groups()) > 1 and match.group(2).isdigit():
                pack_size = int(match.group(2))
                break

    return pack_size

print("Extracting package size from product names...")
# Apply the function only to rows where pack_size is empty
mask_missing_pack_size = df['pack_size'].isna()
df.loc[mask_missing_pack_size, 'pack_size'] = df.loc[mask_missing_pack_size, 'name'].apply(extract_pack_size)

# Convert values to numeric or leave as is
df['pack_size'] = pd.to_numeric(df['pack_size'], errors='ignore')

print(f"Number of non-empty values in pack_size now: {df['pack_size'].notna().sum()}")
print("Sample results after filling:")
print(df[df['pack_size'].notna()][['name', 'pack_size']].head(10))

# Optional step: Fill remaining empty values with 1 (single item)
# This is optional and depends on your analysis. Uncomment the next line if you want to do this.
# df['pack_size'].fillna(1, inplace=True)

# --- 3. Reviewing the keyword column (category) ---
print("\n--- Step 3: Reviewing keyword Column ---")

# Display current unique values
print(f"Current unique values in keyword: {df['keyword'].unique()}")

# Example of fixing the "water" misclassification
# Look for products containing words like "antifreeze" or "radiator" in the name
# and change their category or delete them if they're out of scope
automotive_keywords = ['antifreeze', 'radiator', 'cooling water']
for kw in automotive_keywords:
    mask = df['name'].str.contains(kw, case=False, na=False)
    if mask.any():
        print(f"Found {mask.sum()} product(s) containing '{kw}'. Their current category: {df.loc[mask, 'keyword'].unique()}")
        # You could change the category here, for example:
        # df.loc[mask, 'keyword'] = 'non-food'
        # Or delete them:
        # df.drop(df[mask].index, inplace=True)
        # I'll just print a warning in this example.
        print(f"  Warning: These products might be misclassified under '{df.loc[mask, 'keyword'].iloc[0]}'.")
        print(df.loc[mask, 'name'].tolist())

# --- 4. Cleaning the Brand Name column (simple example) ---
print("\n--- Step 4: Cleaning Brand Name Column ---")

# Display the most common brand names
print("Top 10 most common brand names (before cleaning):")
print(df['Brand Name'].value_counts().head(10))

# Example of merging similar names (you can expand this list)
brand_replacements = {
    'juhayna': 'juhayna',
    'jh': 'juhayna',
    'almarai': 'almarai',
    'lamar': 'lamar',
    'generic': 'generic',
    'unknown': 'generic',
    # Add more here as needed
}

# Uncomment the next line to apply the replacements
# df['Brand Name'] = df['Brand Name'].replace(brand_replacements)

print("\n--- Summary of Results ---")
print(f"Remaining rows: {len(df)}")
print(f"Current columns: {df.columns.tolist()}")
print(f"Non-empty pack_size values: {df['pack_size'].notna().sum()}")
print(f"Non-empty weight_in_grams values: {df['weight_in_grams'].notna().sum()}")

# Save the cleaned data to a new file (optional)
# df.to_csv('final_data_cleaned.csv', index=False)
# print("\nCleaned data saved to 'final_data_cleaned.csv'")

Analyzing weights...
  Illogical value: 0.4 Kilograms -> 0.4 grams. Will ignore.
  Illogical value: 0.7 Kilograms -> 0.7 grams. Will ignore.
  Illogical value: 0.25 Kilograms -> 0.25 grams. Will ignore.
  Illogical value: 0.25 Kilograms -> 0.25 grams. Will ignore.
  Illogical value: 0.3 Kilograms -> 0.3 grams. Will ignore.
  Illogical value: 0.17 Kilograms -> 0.17 grams. Will ignore.
  Illogical value: 0.4 Kilograms -> 0.4 grams. Will ignore.
  Illogical value: 0.22 Kilograms -> 0.22 grams. Will ignore.
  Illogical value: 0.43 Kilograms -> 0.43 grams. Will ignore.
  Illogical value: 0.32 Kilograms -> 0.32 grams. Will ignore.
  Illogical value: 0.23 Kilograms -> 0.23 grams. Will ignore.
  Illogical value: 0.49 Kilograms -> 0.49 grams. Will ignore.
  Illogical value: 0.22 Kilograms -> 0.22 grams. Will ignore.
  Illogical value: 0.22 Kilograms -> 0.22 grams. Will ignore.
  Illogical value: 0.43 Kilograms -> 0.43 grams. Will ignore.
  Illogical value: 0.43 Kilograms -> 0.43 grams. Will ign

In [ ]:
df.shape

(8071, 18)

**Just for test on Linear Regression**

In [ ]:

# الخيار 3: استخدام نموذج للتنبؤ بالوزن المفقود (متقدم)
from sklearn.ensemble import RandomForestRegressor

# تدريب نموذج على الصفوف اللي فيها وزن
mask = df['weight_in_grams'].notna()
train_data = df[mask]
missing_data = df[~mask]

# استخدام ميزات تانية للتنبؤ بالوزن
features_for_weight = ['price', 'pack_size'] + [col for col in df.columns if col.startswith('is_')]
X_train = train_data[features_for_weight]
y_train = train_data['weight_in_grams']
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# التنبؤ بالوزن المفقود
X_missing = missing_data[features_for_weight]
predicted_weights = rf.predict(X_missing)

# ملء القيم المفقودة
df.loc[~mask, 'weight_in_grams'] = predicted_weights


In [ ]:
df['weight_in_grams']

,weight_in_grams
0,1.0
1,1.0
2,2.0
3,1.0
4,1.0
...,...
8066,454.0
8067,125.0
8068,400.0
8069,200.0


In [ ]:
df['weight_in_grams'].isna().sum()

np.int64(0)

In [ ]:

df['pack_size'] = df['pack_size'].fillna(1)


df['pack_size'] = df.groupby('keyword')['pack_size'].transform(
    lambda x: x.fillna(x.median())
).fillna(1)

In [ ]:
df['pack_size']

,pack_size
0,1.0
1,1.0
2,1.0
3,1.0
4,1.0
...,...
8066,1.0
8067,1.0
8068,1.0
8069,1.0


In [ ]:
df['pack_size'].isna().sum()

np.int64(0)

In [ ]:

tfidf_sizes = [200, 500, 1000, 2000]

for size in tfidf_sizes:
    tfidf = TfidfVectorizer(max_features=size, ngram_range=(1, 2))
    name_features = tfidf.fit_transform(df['name'].fillna(''))
    print(f"Size {size}: {name_features.shape}")


Size 200: (8071, 200)
Size 500: (8071, 500)
Size 1000: (8071, 1000)
Size 2000: (8071, 2000)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

for size in tfidf_sizes:
    tfidf = TfidfVectorizer(max_features=size, ngram_range=(1,2))
    X = tfidf.fit_transform(df['name'].fillna(''))
    y = df['price']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression()
    model.fit(X_train, y_train)
    print(f"TF-IDF size {size} model R^2: {model.score(X_test, y_test):.4f}")

TF-IDF size 200 model R^2: 0.2850
TF-IDF size 500 model R^2: 0.3915
TF-IDF size 1000 model R^2: 0.4794
TF-IDF size 2000 model R^2: 0.4411


**The best one at 1000 features**

In [ ]:

tfidf = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

**Make Onehot Encoding**

In [ ]:

def combine_rare_categories(series, threshold=10):
    value_counts = series.value_counts()
    rare_values = value_counts[value_counts < threshold].index
    return series.replace(rare_values, 'other')

df['Flavor'] = combine_rare_categories(df['Flavor'], threshold=10)
df['Item Form'] = combine_rare_categories(df['Item Form'], threshold=5)

**Handle outliers**

In [ ]:

Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df = df[(df['price'] >= lower_bound) & (df['price'] <= upper_bound)]

In [ ]:
def remove_outliers_by_group(df, group_col, value_col):
    def remove_outliers_group(group):
        Q1 = group[value_col].quantile(0.25)
        Q3 = group[value_col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        return group[(group[value_col] >= lower) & (group[value_col] <= upper)]

    return df.groupby(group_col, group_keys=False).apply(remove_outliers_group)

df = remove_outliers_by_group(df, 'keyword', 'price')

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler


numeric_cols = ['price', 'pack_size', 'weight_in_grams'] + \
               [col for col in df.columns if col.startswith('is_')]
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])


**To handle Immbalance in Data**

**Extractions**

In [ ]:

min_samples_per_category = 10

for col in ['keyword', 'Brand Name', 'Flavor', 'Item Form']:
    value_counts = df[col].value_counts()
    rare_values = value_counts[value_counts < min_samples_per_category].index
    df[col] = df[col].replace(rare_values, 'other')

In [ ]:

df['price_per_kg'] = df['price'] / df['weight_in_grams'] * 1000

df = df[(df['price_per_kg'] > 0) & (df['price_per_kg'] < 10000)]

In [ ]:

df['brand_in_name'] = df.apply(
    lambda row: 1 if row['Brand Name'].lower() in str(row['name']).lower() else 0,
    axis=1
)

In [ ]:

df['weight_category'] = pd.cut(
    df['weight_in_grams'],
    bins=[0, 100, 500, 1000, 5000, float('inf')],
    labels=['small', 'medium', 'large', 'xlarge', 'bulk']
)

In [ ]:
flavor_keywords = {
    'chocolate': ['chocolate', 'cocoa', 'شوكولاتة'],
    'strawberry': ['strawberry', 'فراولة'],
    'vanilla': ['vanilla', 'فانيليا'],
    'mango': ['mango', 'مانجو'],

}

def extract_flavor_from_name(row):
    if row['Flavor'] != 'unknown':
        return row['Flavor']

    name_lower = str(row['name']).lower()
    for flavor, keywords in flavor_keywords.items():
        if any(keyword in name_lower for keyword in keywords):
            return flavor
    return 'unknown'

df['Flavor'] = df.apply(extract_flavor_from_name, axis=1)

In [178]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix

# قراءة البيانات النهائية
df = pd.read_csv('/content/final_data_cleaned.csv')



tfidf = TfidfVectorizer(
    max_features=1000,  # جرب 500 أو 1000 أو 2000
    ngram_range=(1, 2),  # unigrams and bigrams
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)
name_features = tfidf.fit_transform(df['clean_name'].fillna(''))

cat_cols = ['keyword', 'Brand Name', 'Flavor', 'Item Form', 'weight_category']
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
cat_features = encoder.fit_transform(df[cat_cols])


num_cols = ['is_organic', 'is_sugar_free', 'is_gluten_free',
            'is_vegan', 'is_keto', 'is_diet', 'is_zero', 'is_stevia',
            'price_per_kg', 'pack_size', 'brand_in_name']



num_features = df[num_cols].values


X_combined = hstack([
    name_features,
    cat_features,
    csr_matrix(num_features)
])


y = df['price_per_kg'].values

print(f"Shape of features: {X_combined.shape}")
print(f"Number of features: {X_combined.shape[1]}")
print(f"Number of samples: {len(y)}")




Shape of features: (1881, 1254)
Number of features: 1254
Number of samples: 1881


In [183]:
import pandas as pd
import re

# Load the CSV file
df = pd.read_csv('/content/final version2')

# Function to extract weight in grams
def extract_weight_grams(weight_str):
    if pd.isna(weight_str):
        return None
    weight_str = str(weight_str)
    # Pattern to find numbers (including decimals) and the unit
    match = re.search(r'(\d+(?:\.\d+)?)\s*(Grams|Gram|g|Kg|Kilograms|ml|Milliliters|Milligrams)', weight_str, re.IGNORECASE)
    if match:
        value = float(match.group(1))
        unit = match.group(2).lower()
        if 'kg' in unit or 'kilogram' in unit:
            return value * 1000
        elif 'g' in unit or 'gram' in unit:
            return value
        elif 'ml' in unit or 'milliliter' in unit:
            # Assuming 1ml = 1g for simplicity, but this might not always be accurate
            return value
        elif 'milligram' in unit:
            return value / 1000
        else:
            return None
    return None

# Apply the function to create the new column
df['actual_weight_gm'] = df['Item Weight'].apply(extract_weight_grams)

# Display the first 20 rows of relevant columns to check the result
# Also showing 'weight_in_grams' for comparison
print(df[['name', 'Item Weight', 'actual_weight_gm', 'weight_in_grams']].head(20))

                                                 name     Item Weight  \
0                hero delicia biscuits spreads 300 gm       300 Grams   
1   hero delicia vanilla biscuit spread with choco...       300 Grams   
2       biscuits (24 pakt - 2 boxes, fairy sadah 50g)             NaN   
3                classic tea biscuits, vanilla - 1 kg     1 Kilograms   
4                         ulker tea biscuits 75g x 12        75 Grams   
5   biscuits (24 packets, candlestick wafer filled...             NaN   
6   oreo chocolate crème filled biscuit - 5 cookie...        43 Grams   
7   oreo hazelnut crème filled biscuits 5 cookies ...     43.75 Grams   
8   fitness biscuits apple & cinnamon 30g - pack o...  0.36 Kilograms   
9   k keto rockets fresh plain tea biscuits - diet...       0.9 Grams   
10  k keto rockets digestive chocolate fresh biscu...       0.9 Grams   
11  k keto rockets digestive plain fresh biscuits ...  0.11 Kilograms   
12  k keto rockets digestive chocolate fresh biscu.

In [185]:
df.to_csv('final3',index=False)

In [179]:
df.to_csv('final version2',index=False)

**Ready To be pushed to DeepLearning Model**

In [180]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")


import joblib


joblib.dump(X_train, 'X_train.pkl')
joblib.dump(X_test, 'X_test.pkl')
joblib.dump(y_train, 'y_train.pkl')
joblib.dump(y_test, 'y_test.pkl')
joblib.dump(tfidf, 'tfidf_model.pkl')
joblib.dump(encoder, 'encoder_model.pkl')

print("✅ Data saved successfully!")

Training set: (1504, 1254)
Test set: (377, 1254)
✅ Data saved successfully!
